# RJMCMC
Checking data stationarity by implementing RJMCMC analysis with change points.


In [ ]:
from torch.utils.data import ConcatDataset
import glob
import xarray as xr

basin_files = glob.glob("Caravan-nc/timeseries/netcdf/camels/*.nc")
basin_files = basin_files[:25]

In [6]:
import numpy as np
from scipy.stats import pearson3

def calc_return_periods(ds):
    '''
    Takes in nc ds and treats it using B17b return period fitting 
    Returns dict (water_year, return period of max streamflow event that year)
    '''

    annual_max = ds['streamflow'].groupby('water_year').max('date')
    annual_max = annual_max.where(annual_max > 0, drop=True)  # remove zero-flow years
    log_annual_max = np.log10(annual_max)

    # annual_max = ds['streamflow'].groupby('water_year').max('date')
    # log_annual_max = np.log10(annual_max) # For the B17b fitting procedure

    params = pearson3.fit(log_annual_max)
    skew, mu, sig = params # shape location scale of the fit

    # Calculate the return years by evaluating the fit PDF distribution
    p3cdf = pearson3.cdf(log_annual_max, skew, loc=mu, scale=sig)
    return_periods_list = 1/(1-p3cdf)

    # I want these to live in a dictionary
    return_periods = dict(zip(annual_max['water_year'].values, return_periods_list))

    return return_periods

def add_water_year(ds):
    '''
    Water year is defined as Oct 1 to  Sep 30.
    Oct, Nov, Dec are labelled as regular year + 1
    '''
    t = ds['date']
    water_year = t.dt.year + (t.dt.month >= 10)
    new_ds = ds.assign_coords(water_year=water_year)
    return new_ds

In [ ]:
from torch.utils.data import ConcatDataset

multibasin_annual_maxima = [] # One entry for each basin

for basin_file in basin_files:
    ds = xr.open_dataset(basin_file)

    # Get rid of NaN values
    ds = ds.dropna(dim='date', 
               subset=['streamflow', 
                       'total_precipitation_sum',
                       'temperature_2m_mean',
                       'surface_net_solar_radiation_mean',
                       'surface_net_thermal_radiation_mean',
                       'dewpoint_temperature_2m_mean',
                       'u_component_of_wind_10m_mean',
                       'v_component_of_wind_10m_mean'], # From which features
                 how='any') # Any of the subsets

    # Water year
    ds = add_water_year(ds)

    # Put the max streamflow values into a list
    basin_annual_maxima = ds['streamflow'].groupby('water_year').max('date')
    basin_annual_maxima = basin_annual_maxima.where(basin_annual_maxima > 0, drop=True)  # remove zero-flow years
    basin_annual_maxima_dict = dict(zip(basin_annual_maxima['water_year'].values, basin_annual_maxima.values))
    multibasin_annual_maxima.append(basin_annual_maxima_dict)


In [ ]:
import numpy as np
from scipy.stats import poisson, norm, halfnorm, uniform
from scipy.special import gammaln, logsumexp
from tqdm import tqdm
from copy import deepcopy
from itertools import chain

class RJMCMC:

    def __init__(self,
                        multibasin_annual_maxima,
                        k_lam = 3, k_max = 10,
                        gam_mu = 0.3, gam_std = 0.6,
                        sig_mu = 0, sig_std = 1,
                 ):
        '''
        Parameters
        ----------
        multibasin_annual_maxima : list of dict
            One dict per basin mapping water_year (int) -> annual maximum streamflow (float).
        k_lam : float
            Poisson prior rate on number of change points k ~ Poisson(k_lam).
        k_max : int
            Maximum number of change points allowed.
        gam_mu, gam_std : float
            Mean and std of the Normal prior on LP3 skewness gamma. Defaults from
            Bulletin 17C cross-basin RMSE (Martins & Stedinger 2000).
        sig_mu, sig_std : float
            Location and scale of the HalfNormal prior on LP3 std sigma (must be positive).
        '''
        self.multibasin_annual_maxima =  [list(basin_annual_maxima.values()) for basin_annual_maxima in multibasin_annual_maxima]
        self.multibasin_key_years = [list(basin_annual_maxima.keys()) for basin_annual_maxima in multibasin_annual_maxima]

        self.k_lam = k_lam
        self.k_max = k_max
        self.gam_mu = gam_mu
        self.gam_std = gam_std
        self.sig_mu = sig_mu
        self.sig_std = sig_std

        vals = [v for d in multibasin_annual_maxima for v in d.values()]
        self.mu_max = max(vals)
        self.mu_min = min(vals)
        self.mu_mean = np.mean(vals)
        self.mu_std = np.std(vals)

        self.prior_k = poisson(self.k_lam)
        self.prior_gam = norm(self.gam_mu, self.gam_std)
        self.prior_sig = halfnorm(self.sig_mu, self.sig_std)
        self.prior_mu = uniform(self.mu_min, self.mu_max)

    def calc_c(self):
        '''
        Compute the scaling constant c that caps birth/death proposal probabilities
        at 0.9 across all k. Setting b_k and d_k proportional to min(1, p(k±1)/p(k))
        causes the prior ratio on k to cancel in the MH acceptance ratio (Green 1995).
        Stores result in self.c.
        '''
        k_vals = np.arange(0, self.k_max + 1)

        b_k = np.array([ min(1, self.prior_k.pmf(k+1)/self.prior_k.pmf(k)) 
                        if k<self.k_max else 0.0 for k in k_vals ])
        d_k = np.array([ min(1, self.prior_k.pmf(k-1)/self.prior_k.pmf(k)) 
                        if k>0 else 0.0 for k in k_vals ])
        
        self.c = 0.9/np.max(b_k + d_k)

    def log_likelihood(self,
                state,
                basin_annual_maxima
            ):
        '''
        LP3 log-likelihood for one basin under the current state.

        Parameters
        ----------
        state : list, length 4k+3
            Encodes k change points followed by (k+1) sets of LP3 parameters:
            [cp_1, ..., cp_k, mu_0, ..., mu_k, sig_0, ..., sig_k, gam_0, ..., gam_k]
            Change points are positional indices into basin_annual_maxima.
        basin_annual_maxima : list of float
            Annual maximum streamflows for a single basin, in water year order.

        Returns
        -------
        float
            Total log-likelihood summed across all k+1 segments.
        '''
        params = state
        k = (len(params) - 3) // 4

        change_points = [0] + params[0:k]
        mus = params[k:2*k+1]
        sigs= params[2*k+1:3*k+2]
        gams = params[3*k+2:4*k+3]

        # Split the stuff by change point
        segments = []
        for i in range(0, len(change_points)-1):
            start = change_points[i]
            end = change_points[i+1]
            segment = basin_annual_maxima[start:end]
            segments.append(segment)
        segments.append(basin_annual_maxima[change_points[-1]:])

        total_lnL = 0
        for idx in range(0,len(segments)):
            segment = segments[idx]
            n = len(segment)
            LN10 = np.log(10)
            mu = mus[idx]
            sig = sigs[idx]
            gam = gams[idx]

            alpha = 4 / (gam * gam)
            beta = (sig * gam) / 2
            chi = mu - ((2*sig)/gam)

            segment_lnL = 0
            segment_lnL += -np.sum(np.log(segment))
            segment_lnL += -n*np.log(LN10) - n*gammaln(alpha)
            segment_lnL += (alpha-1) * np.sum(np.log(np.abs(np.log10(segment) - chi)))
            segment_lnL += -n * alpha * np.log(np.abs(beta))
            segment_lnL += (-1/beta) * np.sum(np.log10(segment)-chi)

            total_lnL += segment_lnL

        return total_lnL

    
    def log_prior(self,
                        mu, sig, gam):
        '''
        Calculates the prior of a state or proposed state
        '''
        lnPi = 0

        lnPi += self.prior_mu.logpdf(mu)
        lnPi += self.prior_sig.logpdf(sig)
        lnPi += self.prior_gam.logpdf(gam)

        return lnPi
        
    def transition(self, state, basin_annual_maxima):
        '''
        MH transition to the next state in the chain for a single basin.

        Parameters
        ----------
        state : list, length 4k+3
            Current state encoding change points and LP3 parameters (see log_likelihood).
        basin_annual_maxima : list of float
            Annual maximum streamflows for this basin, in water year order.

        Returns
        -------
        next_state : list
            Proposed next state (accepted or current state if rejected).
        move_type : int
            0 = LP3 parameter update, 1 = change point position move,
            2 = birth (add change point), 3 = death (remove change point).
        accept : int
            1 if the proposed move was accepted, 0 otherwise.
        '''
        k = (len(state) - 3) // 4

        p_birth = self.c * min(1, self.prior_k.pmf(k+1)/self.prior_k.pmf(k)) \
                if k < self.k_max else 0.0
        p_death = self.c * min(1, self.prior_k.pmf(k-1)/self.prior_k.pmf(k)) \
                if k > 0 else 0.0

        if k == 0:
            p_theta = 1.0 - p_birth - p_death
            p_pos = 0.0
        else:
            p_theta = 0.5 * (1.0 - p_birth - p_death)
            p_pos = 0.5 * (1.0 - p_birth - p_death)

        assert np.isclose(p_theta + p_pos + p_birth + p_death, 1.0), \
            "Move probabilities do not sum to 1."

        # POSSIBLE MOVE TYPES:
        # 0 : p_theta = LP3 parameter update, with probability p_theta
        # 1 : p_pos   = change point position move, with probability p_pos
        # 2 : p_birth = birth move (add change point), with probability p_birth
        # 3 : p_death = death move (remove change point), with probability p_death

        move_type = np.random.choice(4, p=np.array([p_theta, p_pos, p_birth, p_death]))

        if move_type == 0:
            next_state, accept = self.theta_move(state, basin_annual_maxima)
        elif move_type == 1:
            next_state, accept = self.pos_move(state, basin_annual_maxima)  
        elif move_type == 2:
            next_state, accept = self.birth_move(state, basin_annual_maxima)  
        elif move_type == 3:
            next_state, accept = self.death_move(state, basin_annual_maxima)  
        else:
            raise ValueError("Invalid move type.")
        
        return next_state, move_type, accept

    def theta_move(self, state, basin_annual_maxima):
        '''
        This move changes the theta values of the one segment
        '''
        k = (len(state) - 3) // 4

        proposed_state = deepcopy(state)

        # choose which segment theta to alter
        # k changes means k+1 segments
        j = np.random.randint(0,k+1)

        # extract the segment theta
        change_points = [0] + state[0:k]
        current_mu = state[k:2*k+1][j]
        current_sig = state[2*k+1:3*k+2][j]
        current_gam = state[3*k+2:4*k+3][j]

        # propose new theta from the old one
        if j != k: # has not selected the final change point
            start = change_points[j]
            end = change_points[j+1]
            n_seg = end-start # len of segment
            segment = basin_annual_maxima[start:end]
        else:
            start = change_points[j]
            n_seg = len(basin_annual_maxima) - start
            segment = basin_annual_maxima[start:]

        # theta prime
        proposed_mu = current_mu + np.random.normal(0, 0.5 * np.std(np.log10(segment)) / np.sqrt(n_seg))
        proposed_sig = current_sig * np.exp(np.random.normal(0, 0.5 / np.sqrt(2 * n_seg)))
        proposed_gam = current_gam + np.random.normal(0, 0.5 * np.sqrt(6 / n_seg))

        proposed_state[k+j] = proposed_mu
        proposed_state[2*k+1+j] = proposed_sig
        proposed_state[3*k+2+j] = proposed_gam

        # likelihood and prior ratios (log)
        LnLR = self.log_likelihood(proposed_state, basin_annual_maxima) - self.log_likelihood(state, basin_annual_maxima)
        LnPiR = self.log_prior(proposed_mu, proposed_sig, proposed_gam) - self.log_prior(current_mu, current_sig, current_gam)
        LnCorrection = np.log(proposed_sig / current_sig) # because the sigma is multiplicative by an exp so the linearity is in log space
        
        # Acceptance prob
        log_accept_prob = LnLR + LnPiR + LnCorrection

        # Check if accept
        if np.log(np.random.uniform()) < log_accept_prob:
            accept = 1
            return proposed_state, accept
        else:
            accept = 0
            return state, accept

    def pos_move(self, 
                 state,
                 eps = 1e-10):
        